# EDA Project — Red Wine Quality

This is a mini end-to-end EDA (Exploratory Data Analysis) project.

**Dataset:** Red Wine Quality (UCI ML Repository)  
**Question:** What physicochemical properties most influence wine quality?

## EDA Workflow
1. Load & inspect the data
2. Handle missing values
3. Univariate analysis (distributions)
4. Bivariate analysis (correlations, group comparisons)
5. Key findings & takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import urllib.request

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Download data from UCI ML Repository
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
try:
    df = pd.read_csv(URL, sep=';')
    print('Downloaded from UCI')
except Exception:
    # Fallback: generate synthetic data with similar properties
    print('Using synthetic fallback data')
    np.random.seed(42)
    n = 1599
    df = pd.DataFrame({
        'fixed acidity':       np.random.normal(8.32, 1.74, n).clip(4.6, 15.9),
        'volatile acidity':    np.random.normal(0.528, 0.179, n).clip(0.12, 1.58),
        'citric acid':         np.random.normal(0.271, 0.195, n).clip(0, 1),
        'residual sugar':      np.random.exponential(2.5, n).clip(1.2, 15.5),
        'chlorides':           np.random.normal(0.087, 0.047, n).clip(0.012, 0.611),
        'free sulfur dioxide': np.random.normal(15.87, 10.46, n).clip(1, 72),
        'total sulfur dioxide':np.random.normal(46.47, 32.9, n).clip(6, 289),
        'density':             np.random.normal(0.9967, 0.0019, n).clip(0.990, 1.004),
        'pH':                  np.random.normal(3.311, 0.154, n).clip(2.74, 4.01),
        'sulphates':           np.random.normal(0.658, 0.170, n).clip(0.33, 2.0),
        'alcohol':             np.random.normal(10.42, 1.07, n).clip(8.4, 14.9),
        'quality':             np.random.choice([3,4,5,6,7,8], n, p=[0.006,0.033,0.426,0.399,0.124,0.011])
    })

print(df.shape)
print(df.head())

## Step 1 — Inspect the Data

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
print(df.describe().T.round(2))

## Step 2 — Target Variable: Quality Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='quality', palette='Blues_d', ax=axes[0])
axes[0].set_title('Quality Score Distribution')
axes[0].set_xlabel('Quality Score (3–8)')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height()}', (p.get_x()+p.get_width()/2, p.get_height()+5),
                     ha='center', fontsize=9)

# Binary classification view
df['quality_label'] = df['quality'].apply(lambda q: 'Good (>=7)' if q >= 7 else 'Average/Poor (<7)')
df['quality_label'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1],
                                             colors=['steelblue','coral'], startangle=90)
axes[1].set_title('Binary Quality Split')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## Step 3 — Feature Distributions

In [ ]:
features = df.columns.drop(['quality', 'quality_label']).tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

for ax in axes[len(features):]:
    ax.set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4 — Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[features + ['quality']].corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\nTop features correlated with quality:')
print(corr['quality'].drop('quality').sort_values(key=abs, ascending=False).round(3))

## Step 5 — Key Feature vs Quality Analysis

In [ ]:
top_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    sns.boxplot(data=df, x='quality', y=feat, palette='Blues_d', ax=axes[i])
    axes[i].set_title(f'{feat} by Quality Score')
    axes[i].set_xlabel('Quality Score')

plt.suptitle('Top Features vs Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of top features
g = sns.pairplot(df[top_features + ['quality_label']], hue='quality_label',
                 diag_kind='kde', plot_kws={'alpha': 0.4}, height=2.5,
                 palette={'Good (>=7)':'steelblue', 'Average/Poor (<7)':'coral'})
g.figure.suptitle('Top Features — Good vs Average/Poor Wine', y=1.02)
plt.show()

## Step 6 — Statistical Group Comparison

In [ ]:
# Mean of each feature per quality class
means = df.groupby('quality')[features].mean().T
print(means.round(3))

In [ ]:
# Good wines vs Poor wines comparison
good = df[df['quality'] >= 7]
poor = df[df['quality'] <= 4]

comparison = pd.DataFrame({
    'Good wines (≥7)': good[features].mean(),
    'Poor wines (≤4)': poor[features].mean(),
}).round(3)
comparison['Difference'] = (comparison['Good wines (≥7)'] - comparison['Poor wines (≤4)']).round(3)
print(comparison.sort_values('Difference', key=abs, ascending=False))

## Key Findings

1. **Alcohol** has the strongest positive correlation with quality (r ≈ +0.48) — higher alcohol = better wine
2. **Volatile acidity** has the strongest negative correlation (r ≈ -0.39) — high acidity makes wine taste like vinegar
3. **Sulphates** and **citric acid** are positively correlated with quality — act as preservatives/freshness agents
4. Quality scores cluster around 5–6 (average) — only ~14% of wines score ≥7
5. No features follow a perfect normal distribution — several are right-skewed (residual sugar, chlorides)

## Next Steps

With this EDA complete, the next step would be to:
- Encode quality as binary (good/poor) for classification
- Apply feature scaling (StandardScaler)
- Train a Random Forest or Gradient Boosting classifier
- Evaluate with precision, recall, F1

→ See Section 05 (Machine Learning) for the modelling step.